# NEU-DET Transfer Learning Master Notebook

**Goal:** Surpass 85% mAP@0.5 on NEU-DET using a professional 3-stage transfer-learning pipeline.

**Baseline:** YOLOv11n pretrained on COCO.
**Architecture:** YOLOv11n + DAFE (Defect-Aware Feature Enhancement) at P2/P3.
**Dataset:** Preprocessed NEU-DET (CLAHE + 320×320 upscale).
**Loss:** Inner-WIoU via `digisteel.engine.trainer`.

**Pipeline:**
| Stage | Frozen | Trainable | LR | Epochs |
|-------|--------|-----------|-----|--------|
| 1. Warm-up Head | Backbone + Neck (0–22) | Head (23) | 0.001 | 50 |
| 2. Neck Fine-Tune | Backbone (0–10) | Neck + Head (11–23) | 0.0005 | 100 |
| 3. Full Fine-Tune | None | All | 0.0001 | 200 |

## 1. Setup & Reproducibility

In [11]:
import torch
import ultralytics
from pathlib import Path
import json
import random
import numpy as np
from datetime import datetime
import sys

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Paths
ROOT = Path("D:/DigiSteel-Yolo/DigiSteel-YOLO")
sys.path.insert(0, str(ROOT))  # Add project root to Python path for digisteel imports

DATA_YAML = ROOT / "datasets/NEU-DET/yolo_preprocessed/dataset.yaml"
MODEL_YAML = ROOT / "configs/models/digisteel.yaml"
RUNS_DIR = ROOT / "runs/detect"
EVALS_DIR = ROOT / "evals"
EVALS_DIR.mkdir(parents=True, exist_ok=True)

# Experiment name
EXPERIMENT_NAME = "tl_master_yolo11n_dafe"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 70)
print("  NEU-DET TRANSFER LEARNING MASTER")
print("=" * 70)
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  CUDA: {torch.cuda.is_available()}")
print(f"  Ultralytics: {ultralytics.__version__}")
print(f"  Data YAML: {DATA_YAML} (exists: {DATA_YAML.exists()})")
print(f"  Model YAML: {MODEL_YAML} (exists: {MODEL_YAML.exists()})")
print("=" * 70)

  NEU-DET TRANSFER LEARNING MASTER
  Device: NVIDIA RTX 2000 Ada Generation
  CUDA: True
  Ultralytics: 8.4.83
  Data YAML: D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo_preprocessed\dataset.yaml (exists: True)
  Model YAML: D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\models\digisteel.yaml (exists: True)


## 2. Baseline Selection: Compare Pretrained Models

We compare YOLOv11n, YOLOv12n, YOLOv11s, and YOLOv8n on parameter count.
For NEU-DET (1,290 train images), we want enough capacity without overfitting.

In [12]:
from ultralytics import YOLO

baseline_candidates = ["yolo11n.pt", "yolo12n.pt", "yolo11s.pt", "yolov8n.pt"]
baseline_info = []

for m in baseline_candidates:
    model = YOLO(m)
    p = sum(x.numel() for x in model.model.parameters())
    baseline_info.append({"model": m, "params_M": round(p / 1e6, 2)})
    print(f"{m}: {p/1e6:.2f}M params")

# Selected baseline
BASELINE = "yolo11n.pt"
print(f"\nSelected baseline: {BASELINE} (best accuracy/efficiency trade-off for NEU-DET)")

yolo11n.pt: 2.62M params
yolo12n.pt: 2.60M params
yolo11s.pt: 9.46M params
yolov8n.pt: 3.16M params

Selected baseline: yolo11n.pt (best accuracy/efficiency trade-off for NEU-DET)


## 3. Common Training Configuration

Hyperparameters customized for the preprocessed NEU-DET dataset.

In [13]:
def get_base_overrides():
    return {
        "data": str(DATA_YAML),
        "task": "detect",
        "batch": 16,
        "imgsz": 640,
        "device": 0,
        "optimizer": "AdamW",
        "lrf": 0.01,
        "momentum": 0.937,
        "weight_decay": 0.0005,
        "warmup_epochs": 3,
        "warmup_momentum": 0.8,
        "warmup_bias_lr": 0.1,
        "mosaic": 0.0,
        "mixup": 0.15,
        "degrees": 10.0,
        "translate": 0.1,
        "scale": 0.5,
        "shear": 2.0,
        "perspective": 0.0,
        "flipud": 0.0,
        "fliplr": 0.5,
        "hsv_h": 0.0,
        "hsv_s": 0.0,
        "hsv_v": 0.4,
        "erasing": 0.4,
        "cos_lr": True,
        "deterministic": True,
        "close_mosaic": 10,
        "amp": True,
        "seed": SEED,
        "workers": 8,
        "save": True,
        "plots": True,
        "project": str(RUNS_DIR),
        "exist_ok": True,
        "verbose": True,
    }

print("Base overrides configured.")

Base overrides configured.


## 4. Helper Functions for Freeze/Unfreeze

In [14]:
def freeze_layers(model, trainable_indices):
    """
    Freeze all layers except those whose index is in trainable_indices.
    Layer index is extracted from parameter names like 'model.23.conv.weight'.
    """
    for name, param in model.model.named_parameters():
        # Extract layer index from 'model.XX.*'
        parts = name.split(".")
        if len(parts) >= 2 and parts[0] == "model" and parts[1].isdigit():
            idx = int(parts[1])
            param.requires_grad = idx in trainable_indices
        else:
            param.requires_grad = False

def count_trainable(model):
    trainable = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.model.parameters())
    return trainable, total

def print_layer_status(model):
    print(f"{'Idx':<5} {'Status':<12} {'Params':>12}")
    print("-" * 35)
    for i, layer in enumerate(model.model.model.children()):
        nparams = sum(p.numel() for p in layer.parameters())
        if nparams == 0:
            status = "N/A"
        elif all(not p.requires_grad for p in layer.parameters()):
            status = "FROZEN"
        elif any(p.requires_grad for p in layer.parameters()):
            status = "TRAINABLE"
        else:
            status = "MIXED"
        print(f"{i:<5} {status:<12} {nparams:>12,}")

print("Helper functions defined.")

Helper functions defined.


## 5. Stage 1: Warm-up Head

Freeze backbone and neck, train only the detection head with a relatively high learning rate.

In [16]:
from digisteel.engine.trainer import register_custom_modules

register_custom_modules()

stage1_name = f"{EXPERIMENT_NAME}_stage1_head"
stage1_model = YOLO(str(MODEL_YAML))

# Load pretrained COCO weights into backbone/neck/head
# Note: load() transfers compatible layers; DAFE layers are randomly initialized.
stage1_model.load(BASELINE)

# Freeze backbone (0-10) and neck (11-22), train head (23)
freeze_layers(stage1_model, trainable_indices={23})
trainable, total = count_trainable(stage1_model)
print(f"Stage 1 trainable params: {trainable/1e6:.2f}M / {total/1e6:.2f}M")
print_layer_status(stage1_model)

stage1_overrides = get_base_overrides()
stage1_overrides.update({
    "epochs": 50,
    "patience": 20,
    "lr0": 0.001,
    "name": stage1_name,
    "copy_paste": 0.2,
})

print("\nStarting Stage 1: Warm-up Head...")
stage1_results = stage1_model.train(**stage1_overrides)
print(f"Stage 1 complete. Best weights: {RUNS_DIR / stage1_name / 'weights' / 'best.pt'}")

WARNING no model scale passed. Assuming scale='n'.
Transferred 49/373 items from pretrained weights
Stage 1 trainable params: 0.49M / 2.94M
Idx   Status             Params
-----------------------------------
0     FROZEN                464
1     FROZEN              4,672
2     FROZEN              7,360
3     FROZEN              6,273
4     FROZEN             18,560
5     FROZEN             29,056
6     FROZEN             24,833
7     FROZEN             73,984
8     FROZEN            115,456
9     FROZEN            295,424
10    FROZEN            460,288
11    FROZEN            164,608
12    N/A                     0
13    N/A                     0
14    FROZEN            148,224
15    N/A                     0
16    N/A                     0
17    FROZEN             37,248
18    FROZEN             36,992
19    N/A                     0
20    FROZEN            123,648
21    FROZEN            147,712
22    N/A                     0
23    TRAINABLE         493,056
24    FROZEN            

## 6. Stage 2: Neck Fine-Tune

Load Stage 1 weights, freeze backbone, and fine-tune neck + head.

In [ ]:
stage2_name = f"{EXPERIMENT_NAME}_stage2_neck"
stage1_best = RUNS_DIR / stage1_name / "weights" / "best.pt"

# Sanity check: stage1_best must exist before Stage 2
if not stage1_best.exists():
    raise FileNotFoundError(f"Stage 1 weights not found at {stage1_best}. Run Stage 1 first.")

stage2_model = YOLO(str(MODEL_YAML))
stage2_model.load(str(stage1_best))

# Freeze backbone (0-10), train neck (11-22) + head (23)
freeze_layers(stage2_model, trainable_indices=set(range(11, 24)))
trainable, total = count_trainable(stage2_model)
print(f"Stage 2 trainable params: {trainable/1e6:.2f}M / {total/1e6:.2f}M")
print_layer_status(stage2_model)

stage2_overrides = get_base_overrides()
stage2_overrides.update({
    "epochs": 100,
    "patience": 30,
    "lr0": 0.0005,
    "name": stage2_name,
    "copy_paste": 0.2,
})

print("\nStarting Stage 2: Neck Fine-Tune...")
stage2_results = stage2_model.train(**stage2_overrides)
print(f"Stage 2 complete. Best weights: {RUNS_DIR / stage2_name / 'weights' / 'best.pt'}")

WARNING no model scale passed. Assuming scale='n'.
Transferred 373/373 items from pretrained weights
Stage 2 trainable params: 1.15M / 2.94M
Idx   Status             Params
-----------------------------------
0     FROZEN                464
1     FROZEN              4,672
2     FROZEN              7,360
3     FROZEN              6,273
4     FROZEN             18,560
5     FROZEN             29,056
6     FROZEN             24,833
7     FROZEN             73,984
8     FROZEN            115,456
9     FROZEN            295,424
10    FROZEN            460,288
11    TRAINABLE         164,608
12    N/A                     0
13    N/A                     0
14    TRAINABLE         148,224
15    N/A                     0
16    N/A                     0
17    TRAINABLE          37,248
18    TRAINABLE          36,992
19    N/A                     0
20    TRAINABLE         123,648
21    TRAINABLE         147,712
22    N/A                     0
23    TRAINABLE         493,056
24    FROZEN           

## 7. Stage 3: Full Fine-Tune

Load Stage 2 weights and fine-tune all layers with a low learning rate.

In [ ]:
stage3_name = f"{EXPERIMENT_NAME}_stage3_full"
stage2_best = RUNS_DIR / stage2_name / "weights" / "best.pt"

# Sanity check: stage2_best must exist before Stage 3
if not stage2_best.exists():
    raise FileNotFoundError(f"Stage 2 weights not found at {stage2_best}. Run Stage 2 first.")

stage3_model = YOLO(str(MODEL_YAML))
stage3_model.load(str(stage2_best))

# Unfreeze all layers
for param in stage3_model.model.parameters():
    param.requires_grad = True

trainable, total = count_trainable(stage3_model)
print(f"Stage 3 trainable params: {trainable/1e6:.2f}M / {total/1e6:.2f}M")

stage3_overrides = get_base_overrides()
stage3_overrides.update({
    "epochs": 200,
    "patience": 50,
    "lr0": 0.0001,
    "name": stage3_name,
    "copy_paste": 0.2,
})

print("\nStarting Stage 3: Full Fine-Tune...")
stage3_results = stage3_model.train(**stage3_overrides)
print(f"Stage 3 complete. Best weights: {RUNS_DIR / stage3_name / 'weights' / 'best.pt'}")

## 8. Evaluation on Held-Out Test Set

In [ ]:
from ultralytics import YOLO

final_pt = RUNS_DIR / stage3_name / "weights" / "best.pt"
final_model = YOLO(str(final_pt))

metrics = final_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    verbose=True,

)

map50 = float(metrics.box.map50)
map5095 = float(metrics.box.map)
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)

per_class = {}
for cid, cname in final_model.names.items():
    if cid < len(metrics.box.ap50):
        per_class[cname] = float(metrics.box.ap50[cid])

results = {
    "experiment": EXPERIMENT_NAME,
    "timestamp": TIMESTAMP,
    "baseline": BASELINE,
    "model_yaml": str(MODEL_YAML),
    "data_yaml": str(DATA_YAML),
    "mAP50": map50,
    "mAP50_95": map5095,
    "precision": precision,
    "recall": recall,
    "per_class_ap50": per_class,
    "best_weights": str(final_pt),
}

out_path = EVALS_DIR / f"{EXPERIMENT_NAME}_results.json"
out_path.write_text(json.dumps(results, indent=2))

print("=" * 70)
print("  FINAL TEST RESULTS")
print("=" * 70)
print(f"  mAP@0.5:      {map50:.4f} ({map50*100:.1f}%)")
print(f"  mAP@0.5:0.95: {map5095:.4f} ({map5095*100:.1f}%)")
print(f"  Precision:    {precision:.4f}")
print(f"  Recall:       {recall:.4f}")
print("\n  Per-class AP@0.5:")
for cname, ap in sorted(per_class.items()):
    print(f"    {cname:<18} {ap*100:.1f}%")
print(f"\n  Saved: {out_path}")
print("=" * 70)

## 9. Comparison with Baseline

In [ ]:
# Load previous best for comparison
prev_best_path = EVALS_DIR / "fresh_baseline_results.json"
if prev_best_path.exists():
    prev = json.loads(prev_best_path.read_text())
    print("Comparison with previous best (Fresh Baseline):")
    print(f"  Fresh Baseline mAP@0.5: {prev['map50']*100:.1f}%")
    print(f"  Transfer Learning mAP@0.5: {map50*100:.1f}%")
    print(f"  Improvement: +{(map50 - prev['map50'])*100:.1f}%")
else:
    print("No previous baseline results found for comparison.")

## 10. Export to ONNX

In [ ]:
onnx_path = final_model.export(format="onnx", imgsz=640)
print(f"ONNX exported to: {onnx_path}")